In [1]:
import duckdb # importing duckdb for the reconciliation

In [2]:
con = duckdb.connect() # creating connection

In [3]:
# Create a lightweight virtual view in DuckDB directly on top of the cleaned FEWS CSV.
# This avoids data duplication while enabling high-performance SQL query executions on the file
con.execute("""
    CREATE VIEW fews AS 
    SELECT * FROM read_csv_auto('data/output/fews_cleaned.csv')
""")

In [4]:
con.execute("""
    CREATE VIEW wfp AS 
    SELECT * FROM read_csv_auto('data/output/wfp_cleaned.csv')
""")

In [5]:
con.execute("""
    CREATE VIEW commodity AS 
    SELECT * FROM read_csv_auto('mapping/commodity_map.csv')
""")

In [6]:
con.execute("""
    CREATE VIEW unit_conversion AS 
    SELECT * FROM read_csv_auto('mapping/unit_conversion.csv')
""")

In [7]:
con.execute("""
    CREATE VIEW market_map AS 
    SELECT * FROM read_csv_auto('mapping/market_map.csv')
""")

In [8]:
con.execute("SELECT * FROM fews LIMIT 5").df() # get the first five rows in a dataframe format

,country,market,admin_1,longitude,latitude,cpcv2,source_document,price_type,product_source,unit,unit_type,currency,date,commodity,price,price_missing
0,Nigeria,aba,Abia,7.37063,5.10719,P23490AA,Famine Early Warning Systems Network (FEWS NET...,Retail,Local,ea,Item,NGN,2008-07-14,bread,NaN,1
1,Nigeria,aba,Abia,7.37063,5.10719,P23490AA,Famine Early Warning Systems Network (FEWS NET...,Retail,Local,ea,Item,NGN,2008-08-15,bread,NaN,1
2,Nigeria,aba,Abia,7.37063,5.10719,P23490AA,Famine Early Warning Systems Network (FEWS NET...,Retail,Local,ea,Item,NGN,2008-09-15,bread,NaN,1
3,Nigeria,aba,Abia,7.37063,5.10719,P23490AA,Famine Early Warning Systems Network (FEWS NET...,Retail,Local,ea,Item,NGN,2008-12-16,bread,NaN,1
4,Nigeria,aba,Abia,7.37063,5.10719,P23490AA,Famine Early Warning Systems Network (FEWS NET...,Retail,Local,ea,Item,NGN,2009-01-23,bread,NaN,1


In [9]:
con.execute("""
SELECT DISTINCT f.market, w.market 
FROM fews f
JOIN wfp w ON f.market = w.market
LIMIT 15
""").df() # rechecking the overlapped between the markets in fews and wfp

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,market,market_1
0,aba,aba
1,biu,biu
2,damaturu,damaturu
3,gombe,gombe
4,gujungu,gujungu
5,gwandu,gwandu
6,giwa,giwa
7,maiduguri,maiduguri
8,geidam,geidam
9,lagos,lagos


In [10]:
con.execute("""
SELECT DISTINCT f.commodity, c.mapped_name
FROM fews f
JOIN commodity c ON f.commodity = c.fews_commodity 
LIMIT 20
""").df() # rechecking the overlapped between the markets in fews and wfp

,commodity,mapped_name
0,sorghum (white),sorghum (white)
1,cowpeas (brown),cowpeas (brown)
2,gari (white),gari (white)
3,maize grain (white),maize (white)
4,maize grain (yellow),maize (yellow)
5,millet (pearl),millet
6,cowpeas (white),cowpeas (white)
7,gari (yellow),gari (yellow)
8,groundnuts (shelled),groundnuts (shelled)
9,rice (milled),rice (milled)


In [13]:
# Logic Breakdown:
#   1. Resolves naming variations via pre-built entity mapping configurations.
#   2. Standardizes disparate volumetric prices into a uniform metric (Price/KG).
#   3. Executes a fuzzy chronological spatial join (bounded by a +/- 30 day window).
#   4. Evaluates variance thresholds to categorize records into actionable segments:
#      - 'GAP': Missing cross-agency visibility.
#      - 'MATCH': Acceptable variance threshold adherence (<= 15%).
#      - 'CONFLICT': Material data discrepancy (> 15% price delta).
reconciliation = con.execute("""
WITH fews_mapped AS (
    SELECT fews.*, commodity.mapped_name
    FROM fews
    JOIN commodity ON fews.commodity = commodity.fews_commodity
),
wfp_mapped AS(
    SELECT wfp.*, commodity.mapped_name
    FROM wfp
    JOIN commodity ON wfp.commodity = commodity.wfp_commodity
),
reconciled AS (
    SELECT
        f.market,
        f.mapped_name AS commodity,
        f.price_type,
        f.date AS fews_date,
        w.date AS wfp_date,
        f.price AS fews_price,
        w.price AS wfp_price,
        f.unit AS fews_unit,
        w.unit AS wfp_unit,
        
        -- Normalise prices using unit conversion table
        f.price / COALESCE(uc_f.conversion_factor, 1) AS fews_price_per_kg,
        w.price / COALESCE(uc_w.conversion_factor, 1) AS wfp_price_per_kg
    FROM fews_mapped f
    LEFT JOIN wfp_mapped w
        ON f.market = w.market
        AND f.mapped_name = w.mapped_name
        AND f.price_type = w.price_type
        AND ABS(DATEDIFF('day', f.date, w.date)) <= 30
    LEFT JOIN unit_conversion uc_f ON f.unit = uc_f.unit
    LEFT JOIN unit_conversion uc_w ON w.unit = uc_w.unit
    WHERE f.price IS NOT NULL 
),
classified AS (
    SELECT
        *,
        -- Calculate price delta percentage
        
        CASE
            WHEN wfp_price IS NULL THEN NULL
            WHEN wfp_price_per_kg IS NULL THEN NULL
            ELSE ROUND(ABS(fews_price_per_kg - wfp_price_per_kg)
                 / NULLIF(fews_price_per_kg, 0) * 100, 2)
        END AS delta_pct,
        -- Classify each row
        
        CASE
            WHEN wfp_price IS NULL THEN 'GAP'
            WHEN fews_price IS NULL THEN 'GAP'
            WHEN ABS(fews_price_per_kg - wfp_price_per_kg)
                 / NULLIF(fews_price_per_kg, 0) * 100 <= 15 THEN 'MATCH'
            ELSE 'CONFLICT'
        END AS status
    FROM reconciled
)
SELECT * FROM classified
""").df()

reconciliation

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,market,commodity,price_type,fews_date,wfp_date,fews_price,wfp_price,fews_unit,wfp_unit,fews_price_per_kg,wfp_price_per_kg,delta_pct,status
0,giwa,rice (milled),Retail,2012-05-02,NaT,106.66,NaN,kg,None,106.66,NaN,NaN,GAP
1,giwa,rice (milled),Retail,2012-05-09,NaT,106.66,NaN,kg,None,106.66,NaN,NaN,GAP
2,giwa,rice (milled),Retail,2012-05-16,NaT,106.66,NaN,kg,None,106.66,NaN,NaN,GAP
3,giwa,rice (milled),Retail,2012-05-23,NaT,106.66,NaN,kg,None,106.66,NaN,NaN,GAP
4,giwa,rice (milled),Retail,2012-05-30,NaT,106.66,NaN,kg,None,106.66,NaN,NaN,GAP
...,...,...,...,...,...,...,...,...,...,...,...,...,...
367966,saminaka,gari (white),Wholesale,2022-07-27,2022-07-15,24000.00,24290.32,100 kg,100 kg,240.00,242.9032,1.21,MATCH
367967,saminaka,gari (white),Retail,2022-08-03,2022-07-15,416.67,421.71,kg,kg,416.67,421.7100,1.21,MATCH
367968,saminaka,gari (white),Wholesale,2022-08-03,2022-07-15,24000.00,24290.32,100 kg,100 kg,240.00,242.9032,1.21,MATCH
367969,saminaka,gari (white),Retail,2022-08-10,2022-07-15,447.92,421.71,kg,kg,447.92,421.7100,5.85,MATCH


In [14]:
reconciliation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367971 entries, 0 to 367970
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   market             367971 non-null  object        
 1   commodity          367971 non-null  object        
 2   price_type         367971 non-null  object        
 3   fews_date          367971 non-null  datetime64[us]
 4   wfp_date           244657 non-null  datetime64[us]
 5   fews_price         367971 non-null  float64       
 6   wfp_price          244657 non-null  float64       
 7   fews_unit          367971 non-null  object        
 8   wfp_unit           244657 non-null  object        
 9   fews_price_per_kg  367971 non-null  float64       
 10  wfp_price_per_kg   244657 non-null  float64       
 11  delta_pct          244657 non-null  float64       
 12  status             367971 non-null  object        
dtypes: datetime64[us](2), float64(5), object(6)


In [15]:
reconciliation['status'].value_counts()

status
MATCH       211780
GAP         123314
CONFLICT     32877
Name: count, dtype: int64

In [16]:
reconciliation.to_csv('data/output/reconciled.csv', index=False)
print("Saved reconciled.csv")

Saved reconciled.csv
